In [ ]:
import os
from pathlib import Path
from openai import OpenAI

# Set OPENAI_API_KEY as an environment variable before running, e.g.
#   export OPENAI_API_KEY=sk-...
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
output_dir = Path(os.environ.get("OUTPUT_DIR", "."))
speech_file_path = output_dir / "somber.mp3"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Today is a wonderful day to build something people love!",
    instructions="Speak in a somber tone.",
) as response:
    response.stream_to_file(speech_file_path)

In [ ]:
import os
import pandas as pd

features_raw = pd.read_csv(os.environ.get("FEATURES_RAW_CSV", "../../dataset/features_raw_251001.csv"))
features_norm = pd.read_csv(os.environ.get("AUDIO_CLUSTER_CSV", "../../dataset/audio_cluster_54_k5_v1.csv"))
audio_info = features_norm[["keycode", "kmeans_cluster", "mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0", "F1", "F2",
                        "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
                        "arousal", "valence", "std_arousal", "std_valence"]]

columns_to_suffix = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0", "F1", "F2",
                        "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
                        "arousal", "valence", "std_arousal", "std_valence"]
suffix = "_norm"

df = audio_info.rename(columns={col: col + suffix for col in audio_info.columns if col in columns_to_suffix})

combined = features_raw.merge(df, on="keycode",how="inner")

In [38]:
summary = combined.groupby("kmeans_cluster").mean(columns_to_suffix).reset_index()

In [ ]:
speech_file_path = output_dir / "test4.mp3"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Today is a wonderful day to build something people love!",
    instructions="Speak with mean f0 around 197.10Hz, \
                f0 standard deviation 33.61Hz \
                speech rate 3.58 words per second\
                mean F1 672.73Hz \
                mean F2 1773.45Hz \
                mean MFCC0  -0.247 ",
) as response:
    response.stream_to_file(speech_file_path)

In [ ]:
for i in range(0,5):
    cluster_id = i

    speech_file_path = output_dir / f"sentences{cluster_id}.mp3"

    row = summary.loc[cluster_id, ["mean_f0", "std_f0", "speaking_rate_w", "VUV", "F1", "F2", "MFCC0",
                        "arousal", "valence", "std_arousal", "std_valence"]]

    f0, f0_std, sr, VUV, f1, f2, MFCC, a, v, a_std, v_std = row

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input="Good afternoon ladies and gentlemen. Today is a wonderful day to build something people love. Thank you for listening.",
        instructions=f"Speak with mean f0 around {f0}Hz, \
                    f0 standard deviation {f0_std}Hz \
                    speech rate {sr} words per second\
                    voiced and unvoiced ratio {VUV} \
                    mean F1 {f1}Hz \
                    mean F2 {f2}Hz \
                    mean MFCC0  {MFCC} \
                    mean arousal {a} \
                    mean valence {v} \
                    arousal standard deviation {a_std}\
                    valence standard deviation {v_std}",
    ) as response:
        response.stream_to_file(speech_file_path)

In [ ]:
for i in range(0,5):
    cluster_id = i

    speech_file_path = output_dir / f"sentences{cluster_id}_norm.mp3"

    row = summary.loc[cluster_id, ["mean_f0_norm", "std_f0_norm", "speaking_rate_w_norm", "VUV_norm", "F1_norm", "F2_norm", "MFCC0_norm",
                        "arousal_norm", "valence_norm", "std_arousal_norm", "std_valence_norm"]]

    f0, f0_std, sr, VUV, f1, f2, MFCC, a, v, a_std, v_std = row

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input="Good afternoon ladies and gentlemen. Today is a wonderful day to build something people love. Thank you for listening.",
        instructions=f"I have five groups, and the following values are z-score transformed, \
                    speak with mean f0 around {f0}, \
                    f0 standard deviation {f0_std}, \
                    speech rate {sr} words per second, \
                    voiced and unvoiced ratio {VUV}, \
                    mean F1 {f1}, \
                    mean F2 {f2}, \
                    mean MFCC0  {MFCC}, \
                    mean arousal {a}, \
                    mean valence {v}, \
                    arousal standard deviation {a_std}, \
                    valence standard deviation {v_std}",
    ) as response:
        response.stream_to_file(speech_file_path)

In [40]:
row = summary.loc[3, ["mean_f0", "std_f0", "speaking_rate_w", "F1", "F2", "MFCC0",
                       "arousal", "valence", "std_arousal", "std_valence"]]

f0, f0_std, sr, f1, f2, MFCC, a, v, a_std, v_std = row
print(row)

mean_f0             159.457879
std_f0               17.006357
speaking_rate_w       3.514233
F1                  736.179499
F2                 1949.116670
MFCC0                -0.266941
arousal               0.000566
valence              -0.092852
std_arousal           0.111097
std_valence           0.231884
Name: 3, dtype: float64
